## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

# Financial TDA imports
from financial_tda.topology.embedding import takens_embedding
from financial_tda.topology.filtration import compute_persistence_vr, compute_bottleneck_distance

# TTK visualization imports
from financial_tda.viz.ttk_plots import plot_persistence_curve_comparison, plot_bottleneck_distance_matrix

# Set random seed for reproducibility
np.random.seed(42)

print("Imports successful!")

## Define Market Regimes

We'll fetch S&P 500 data for multiple crisis and normal periods.

In [ ]:
# Define regime periods
regimes = {
    "Crisis 2008": ("2008-09-01", "2008-10-31"),  # Lehman collapse
    "Crisis 2015": ("2015-08-01", "2015-09-30"),  # Flash crash
    "Crisis 2020": ("2020-03-01", "2020-04-30"),  # COVID crash
    "Crisis 2022": ("2022-06-01", "2022-07-31"),  # Inflation
    "Normal 2007": ("2007-04-01", "2007-06-30"),  # Pre-crisis normal
}

print("Regime periods defined:")
for name, (start, end) in regimes.items():
    print(f"  {name}: {start} to {end}")

## Fetch Market Data

In [ ]:
def fetch_returns(symbol, start_date, end_date):
    """Fetch log returns for a given period."""
    data = yf.download(symbol, start=start_date, end=end_date, progress=False)
    prices = data["Adj Close"].values
    log_returns = np.diff(np.log(prices))
    return log_returns


# Fetch data for all regimes
returns_data = {}
symbol = "^GSPC"  # S&P 500

print(f"Fetching data for {symbol}...")
for name, (start, end) in regimes.items():
    returns = fetch_returns(symbol, start, end)
    returns_data[name] = returns
    print(f"  {name}: {len(returns)} returns fetched")

print("\nData fetching complete!")

## Compute Takens Embeddings

Transform time series into point clouds using delay embeddings.

In [ ]:
# Embedding parameters
DELAY = 5
DIMENSION = 3

embeddings = {}

print(f"Creating Takens embeddings (delay={DELAY}, dimension={DIMENSION})...")
for name, returns in returns_data.items():
    embedding = takens_embedding(returns, delay=DELAY, dimension=DIMENSION)
    embeddings[name] = embedding
    print(f"  {name}: {embedding.shape[0]} points")

print("\nEmbeddings complete!")

## Compute Persistence Diagrams

Extract topological features using Vietoris-Rips persistence.

In [ ]:
diagrams = {}

print("Computing persistence diagrams...")
for name, embedding in embeddings.items():
    diagram = compute_persistence_vr(embedding, homology_dimensions=(0, 1, 2))
    diagrams[name] = diagram

    # Count features by dimension
    n_h0 = np.sum(diagram[:, 2] == 0)
    n_h1 = np.sum(diagram[:, 2] == 1)
    n_h2 = np.sum(diagram[:, 2] == 2)

    print(f"  {name}: {len(diagram)} features (H0: {n_h0}, H1: {n_h1}, H2: {n_h2})")

print("\nPersistence computation complete!")

## Persistence Curve Comparison

Compare persistence distributions across regimes using TTK-enhanced visualization.

In [ ]:
# Select diagrams for comparison
comparison_names = ["Crisis 2008", "Crisis 2020", "Normal 2007"]
comparison_diagrams = [diagrams[name] for name in comparison_names]

# Plot persistence curve comparison
fig = plot_persistence_curve_comparison(
    comparison_diagrams,
    labels=comparison_names,
    dimensions=[0, 1],  # Focus on H0 and H1
    figsize=(16, 5),
    title="Market Regime Comparison: Crisis vs Normal",
    highlight_differences=True,
)

plt.show()

print("\nInterpretation:")
print("- Crisis regimes typically show more H1 features (loops) indicating cyclical patterns")
print("- Normal markets have fewer, shorter-lived features")
print("- Higher persistence values indicate more significant topological structure")

## H1 Features Only (Loops)

Focus on 1-dimensional homology (loops) which often indicate cyclical market behavior.

In [ ]:
# Compare all regimes using only H1 features
all_diagrams = [diagrams[name] for name in regimes.keys()]
all_labels = list(regimes.keys())

fig = plot_persistence_curve_comparison(
    all_diagrams,
    labels=all_labels,
    dimensions=[1],  # H1 only
    figsize=(16, 5),
    title="H1 (Loops) Comparison Across All Regimes",
    highlight_differences=False,
)

plt.show()

print("\nObservations:")
print("- Crisis periods cluster together with similar H1 persistence distributions")
print("- Normal market shows distinct pattern with fewer loops")

## Bottleneck Distance Matrix

Compute pairwise distances to identify similar regimes.

In [ ]:
# Plot distance matrix with hierarchical clustering
fig = plot_bottleneck_distance_matrix(
    all_diagrams,
    labels=all_labels,
    dimension=1,  # H1 features
    metric="bottleneck",
    figsize=(12, 10),
    add_dendrogram=True,
)

plt.show()

print("\nDistance Matrix Interpretation:")
print("- Smaller distances indicate similar topological structure")
print("- Dendrogram shows hierarchical clustering of regimes")
print("- Crisis periods should cluster together")

## Wasserstein Distance Comparison

Alternative distance metric for sensitivity analysis.

In [ ]:
# Compare using Wasserstein distance (faster, different sensitivity)
fig = plot_bottleneck_distance_matrix(
    all_diagrams,
    labels=all_labels,
    dimension=1,
    metric="wasserstein",
    figsize=(10, 9),
    add_dendrogram=False,  # Simpler visualization
)

plt.show()

print("\nBottleneck vs Wasserstein:")
print("- Bottleneck: Max distance between matched features (robust to outliers)")
print("- Wasserstein: Total transport cost (sensitive to all features)")

## Quantitative Regime Similarity Analysis

In [ ]:
# Compute all pairwise distances
n_regimes = len(all_diagrams)
distances = np.zeros((n_regimes, n_regimes))

for i in range(n_regimes):
    for j in range(n_regimes):
        distances[i, j] = compute_bottleneck_distance(all_diagrams[i], all_diagrams[j], dimension=1)

# Create DataFrame for easier analysis
dist_df = pd.DataFrame(distances, index=all_labels, columns=all_labels)

print("Bottleneck Distance Matrix (H1 features):")
print(dist_df.round(3))
print("\n")

# Find most similar regimes
print("Most Similar Regime Pairs:")
for i in range(n_regimes):
    for j in range(i + 1, n_regimes):
        if distances[i, j] < 0.1:  # Threshold for "similar"
            print(f"  {all_labels[i]} <-> {all_labels[j]}: {distances[i, j]:.4f}")

# Find most different regimes
print("\nMost Different Regime Pairs:")
max_dist = 0
max_pair = None
for i in range(n_regimes):
    for j in range(i + 1, n_regimes):
        if distances[i, j] > max_dist:
            max_dist = distances[i, j]
            max_pair = (i, j)

if max_pair:
    i, j = max_pair
    print(f"  {all_labels[i]} <-> {all_labels[j]}: {max_dist:.4f}")

## Summary and Conclusions

### Key Findings:

1. **Persistence Curves**: Crisis regimes show distinct persistence distributions compared to normal markets
2. **H1 Features**: Loops indicate cyclical market behavior, more prevalent during crises
3. **Distance Metrics**: Bottleneck distance provides robust regime classification
4. **Clustering**: Crisis periods cluster together, distinct from normal markets

### Applications:

- **Regime Detection**: Use distance metrics for real-time regime identification
- **Risk Management**: Higher H1 persistence may indicate increased volatility
- **Portfolio Allocation**: Adjust strategies based on detected regime
- **Stress Testing**: Use historical crisis topologies for scenario analysis

### Next Steps:

- Integrate with backtest framework (`financial_tda.analysis.backtest`)
- Test with more asset classes (commodities, crypto, bonds)
- Develop real-time regime monitoring dashboard
- Combine with ML models for regime prediction